# 03 · Special methods

**Day 1 · about 45 minutes live**

Why does `len(df)` work on a DataFrame? Why can you write `tn, fp, fn, tp = confusion_matrix(y, p).ravel()`, or `pipeline[0]` on a scikit-learn `Pipeline`? None of this is magic. Python translates the syntax into calls of methods with double underscores, such as `__len__` or `__getitem__`. When your own classes define these methods, they plug into the same syntax.

In this notebook we give the `ConfusionMatrix` from notebook 01 a proper set of special methods. We also build two small objects a churn project really needs: a lookup for a batch of customers, and a business cost metric that you can call like a function.

**Learning goals.** By the end you can:

- explain what a special ("dunder") method is and which syntax triggers which method [Understand, level 1]
- give a class a useful `__repr__`, and make it work with `len()`, `[]`, `for`, `==` and `+` [Apply, level 2]
- write a callable object with `__call__` and use it wherever a function is expected [Apply, level 2]
- judge when overloading an operator helps readers and when it hides meaning [Analyse, level 3]

**Before you start:** notebooks 01 (classes) and 02 (inheritance).

Exercises come in three levels: **Core** (everyone), **Stretch** (if you finish early) and **Challenge** (for experienced programmers). Check cells fail until you solve the exercise above them. That is expected.

## Setup: data, a simple model, and the ConfusionMatrix from notebook 01

We need real predictions to work with. The model is deliberately simple (a logistic regression on five columns). The point of this notebook is the classes around it, not the model.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

DATA_PATH = Path("../data/raw/telco_churn.csv")

raw = pd.read_csv(DATA_PATH)
# TotalCharges is text and blank for customers who joined this month (tenure 0): they owe 0 so far
total_charges = pd.to_numeric(raw["TotalCharges"].str.strip(), errors="coerce").fillna(0.0)

X = pd.DataFrame(
    {
        "tenure": raw["tenure"],
        "MonthlyCharges": raw["MonthlyCharges"],
        "TotalCharges": total_charges,
        "SeniorCitizen": raw["SeniorCitizen"],
        "month_to_month": (raw["Contract"] == "Month-to-month").astype(int),
    }
).set_index(raw["customerID"])
y = pd.Series((raw["Churn"] == "Yes").astype(int).values, index=X.index, name="churn")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
model = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_train, y_train)
churn_proba = model.predict_proba(X_test)[:, 1]
y_pred = (churn_proba >= 0.5).astype(int)
print(f"{len(X_test)} test customers, {y_test.sum()} of them churned")

This is the `ConfusionMatrix` from notebook 01, in compact form. The constructor takes the four counts in the order scikit-learn uses (`tn, fp, fn, tp`).

In [ ]:
class ConfusionMatrix:
    """Counts of a binary classifier's hits and misses (1 = churn)."""

    def __init__(self, tn: int, fp: int, fn: int, tp: int):
        self.tn = tn
        self.fp = fp
        self.fn = fn
        self.tp = tp

    @classmethod
    def from_predictions(cls, y_true, y_pred) -> "ConfusionMatrix":
        t = np.asarray(y_true)
        p = np.asarray(y_pred)
        return cls(
            tn=int(((t == 0) & (p == 0)).sum()),
            fp=int(((t == 0) & (p == 1)).sum()),
            fn=int(((t == 1) & (p == 0)).sum()),
            tp=int(((t == 1) & (p == 1)).sum()),
        )

    @property
    def precision(self) -> float:
        return self.tp / (self.tp + self.fp) if self.tp + self.fp else 0.0

    @property
    def recall(self) -> float:
        return self.tp / (self.tp + self.fn) if self.tp + self.fn else 0.0


cm = ConfusionMatrix.from_predictions(y_test, y_pred)
cm

That last line shows something like `<__main__.ConfusionMatrix at 0x7f...>`. Correct, but useless. Fixing that is our first special method.

## 1. What special methods are

A special method is a method whose name starts and ends with two underscores ("dunder" = double underscore). You almost never call them yourself. Python calls them when you use a piece of syntax on your object:

| You write | Python calls |
|---|---|
| `obj` in the REPL or last line of a cell, `repr(obj)` | `obj.__repr__()` |
| `print(obj)`, `str(obj)`, f-string `f"{obj}"` | `obj.__str__()` (falls back to `__repr__`) |
| `len(obj)` | `obj.__len__()` |
| `obj[key]` | `obj.__getitem__(key)` |
| `key in obj` | `obj.__contains__(key)` (falls back to iterating) |
| `for x in obj`, `a, b = obj`, `list(obj)` | `obj.__iter__()` |
| `a == b` | `a.__eq__(b)` |
| `a + b` | `a.__add__(b)`, and if that gives up, `b.__radd__(a)` |
| `obj(...)` | `obj.__call__(...)` |
| `with obj as x:` | `obj.__enter__()` and later `obj.__exit__(...)` |
| `if obj:` | `obj.__bool__()` (falls back to `__len__`) |

You already met one: `__init__` runs when you write `ConfusionMatrix(...)`. The full list is called the Python **data model**. pandas, NumPy and scikit-learn are built on it, which is why their objects feel like built-in types.

## 2. `__repr__` and `__str__`: how an object shows itself

- `__repr__` is for developers: it should say unambiguously what the object is. Convention: look like the code that would create it.
- `__str__` is for readers of reports: whatever is most readable. If you define only `__repr__`, `print()` uses it too.

In [ ]:
class ConfusionMatrix:
    """Counts of a binary classifier's hits and misses (1 = churn)."""

    def __init__(self, tn: int, fp: int, fn: int, tp: int):
        self.tn = tn
        self.fp = fp
        self.fn = fn
        self.tp = tp

    @classmethod
    def from_predictions(cls, y_true, y_pred) -> "ConfusionMatrix":
        t = np.asarray(y_true)
        p = np.asarray(y_pred)
        return cls(
            tn=int(((t == 0) & (p == 0)).sum()),
            fp=int(((t == 0) & (p == 1)).sum()),
            fn=int(((t == 1) & (p == 0)).sum()),
            tp=int(((t == 1) & (p == 1)).sum()),
        )

    @property
    def precision(self) -> float:
        return self.tp / (self.tp + self.fp) if self.tp + self.fp else 0.0

    @property
    def recall(self) -> float:
        return self.tp / (self.tp + self.fn) if self.tp + self.fn else 0.0

    def __repr__(self) -> str:
        return f"ConfusionMatrix(tn={self.tn}, fp={self.fp}, fn={self.fn}, tp={self.tp})"

    def __str__(self) -> str:
        return (
            f"              pred=0  pred=1\n"
            f"actual=0  {self.tn:>8} {self.fp:>7}\n"
            f"actual=1  {self.fn:>8} {self.tp:>7}"
        )


cm = ConfusionMatrix.from_predictions(y_test, y_pred)
print(cm)  # __str__
print()
print(repr(cm))  # __repr__
cm  # the notebook displays the last expression with __repr__

A good `__repr__` pays off every time you debug: in error messages, in the variable explorer, in a list of objects (`[cm, cm]` uses `__repr__` for each element).

### Check your understanding

```python
results = [ConfusionMatrix(1, 2, 3, 4)]
print(results)
```

Does this print the table from `__str__` or the one-liner from `__repr__`?

<details><summary>Answer</summary>

The one-liner: `[ConfusionMatrix(tn=1, fp=2, fn=3, tp=4)]`. Containers always show their elements with `repr()`, even when you `print` the container. That is one more reason to always write a `__repr__` first.
</details>

## 3. Container behaviour: `__len__`, `__getitem__`, `__contains__`

Each month the retention team receives a file of active customers to score. A small class wrapping that DataFrame makes the common questions read naturally: *how many customers are in the batch?*, *show me this customer*, *is this customer in the batch?*

In [ ]:
class CustomerBatch:
    """Customers to be scored, indexed by customer ID."""

    def __init__(self, customers: pd.DataFrame, month: str):
        self.customers = customers
        self.month = month

    def __len__(self) -> int:
        return len(self.customers)

    def __getitem__(self, customer_id: str) -> pd.Series:
        return self.customers.loc[customer_id]

    def __contains__(self, customer_id: str) -> bool:
        return customer_id in self.customers.index

    def __repr__(self) -> str:
        return f"CustomerBatch(month={self.month!r}, customers={len(self)})"


batch = CustomerBatch(X_test, month="2026-11")
print(batch)
print(len(batch), "customers")
first_id = X_test.index[0]
print(first_id in batch, "0000-NOONE" in batch)
batch[first_id]

Notice how little code that is: every method delegates to the DataFrame. The value is in the interface. Code using `CustomerBatch` reads like the business question, and nobody has to remember that the ID lives in the index.

### Check your understanding

`CustomerBatch` defines `__len__` but not `__bool__`. What does this print?

```python
empty = CustomerBatch(X_test.iloc[:0], month="2026-12")
print("nothing to score" if not empty else "scoring...")
```

<details><summary>Answer</summary>

`nothing to score`. Without `__bool__`, Python asks `__len__`, and a length of 0 counts as false. This is the same rule that makes an empty list false.
</details>

## 4. `__iter__`, `__eq__` and `__add__`: making ConfusionMatrix feel like a value

Three more things a confusion matrix should support:

1. **Unpacking.** scikit-learn users write `tn, fp, fn, tp = confusion_matrix(y, p).ravel()`. With `__iter__`, our class supports the same line. The simplest way to write `__iter__` is to return an iterator over a tuple.
2. **Equality.** Two matrices with the same four counts describe the same result, so `==` should compare counts. Without `__eq__`, Python compares identity (`is`): two separately created objects are never equal.
3. **Addition.** Counts can be pooled. If the model scores customers in November and in December, the confusion matrix for both months together is the sum of the two monthly matrices. The same holds for cross-validation folds. Here `+` has an obvious meaning, so overloading it is a good idea.

`__add__` returns `NotImplemented` (a special constant, not an error) when it does not know how to add the other object. Python then tries the other object's `__radd__`, and if that also gives up it raises a `TypeError` with a clear message.

In [ ]:
class ConfusionMatrix:
    """Counts of a binary classifier's hits and misses (1 = churn)."""

    def __init__(self, tn: int, fp: int, fn: int, tp: int):
        self.tn = tn
        self.fp = fp
        self.fn = fn
        self.tp = tp

    @classmethod
    def from_predictions(cls, y_true, y_pred) -> "ConfusionMatrix":
        t = np.asarray(y_true)
        p = np.asarray(y_pred)
        return cls(
            tn=int(((t == 0) & (p == 0)).sum()),
            fp=int(((t == 0) & (p == 1)).sum()),
            fn=int(((t == 1) & (p == 0)).sum()),
            tp=int(((t == 1) & (p == 1)).sum()),
        )

    @property
    def n(self) -> int:
        return self.tn + self.fp + self.fn + self.tp

    @property
    def precision(self) -> float:
        return self.tp / (self.tp + self.fp) if self.tp + self.fp else 0.0

    @property
    def recall(self) -> float:
        return self.tp / (self.tp + self.fn) if self.tp + self.fn else 0.0

    def __repr__(self) -> str:
        return f"ConfusionMatrix(tn={self.tn}, fp={self.fp}, fn={self.fn}, tp={self.tp})"

    def __iter__(self):
        return iter((self.tn, self.fp, self.fn, self.tp))

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, ConfusionMatrix):
            return NotImplemented
        return tuple(self) == tuple(other)

    def __add__(self, other: object) -> "ConfusionMatrix":
        if not isinstance(other, ConfusionMatrix):
            return NotImplemented
        return ConfusionMatrix(
            tn=self.tn + other.tn,
            fp=self.fp + other.fp,
            fn=self.fn + other.fn,
            tp=self.tp + other.tp,
        )


# Pretend the test customers arrived in two monthly batches
half = len(X_test) // 2
november = ConfusionMatrix.from_predictions(y_test[:half], y_pred[:half])
december = ConfusionMatrix.from_predictions(y_test[half:], y_pred[half:])
both_months = november + december

print("November:", november)
print("December:", december)
print("Pooled:  ", both_months)
print(
    "Same as computing on all test customers at once?",
    both_months == ConfusionMatrix.from_predictions(y_test, y_pred),
)

tn, fp, fn, tp = both_months  # unpacking uses __iter__
print(f"We caught {tp} churners and missed {fn}.")

In [ ]:
# Adding something that is not a ConfusionMatrix fails loudly, thanks to NotImplemented
try:
    both_months + 1
except TypeError as exc:
    print("TypeError:", exc)

### A side effect of `__eq__`: the object is no longer hashable

When a class defines `__eq__`, Python sets `__hash__` to `None`. Objects of that class can no longer be dictionary keys or set members:

In [ ]:
try:
    {november}
except TypeError as exc:
    print("TypeError:", exc)

This is deliberate. A dictionary finds keys by their hash, and equal objects must have equal hashes. Our matrix is **mutable**: if you changed `november.tp` after using it as a key, its hash would have to change and the dictionary would lose track of it. The clean fix is to make the class immutable. A frozen dataclass (notebook 07) does exactly that and generates `__eq__` and `__hash__` for you. The finished version in `src/churn/evaluation.py` is such a frozen dataclass.

**Value or identity?** Equality by value is right for *values*: a confusion matrix, a config, a date. It is wrong for *things with identity*, such as a fitted model. Two different models with the same accuracy are not the same model, so a model class should not define `__eq__` on its metrics.

## 5. `__call__`: objects that behave like functions

Precision and recall do not say what a model is worth to the business. Suppose the retention team sends an offer to every customer the model flags:

- each offer costs **50** (discount plus call-centre time),
- a churner who receives the offer stays with probability **0.3**,
- every churner who leaves costs **600** (roughly a year of margin).

These figures are illustrative assumptions. Plug in your own numbers in a real project.

A cost function needs these three parameters *and* the usual `(y_true, y_pred)` arguments. An object with `__call__` keeps the parameters as attributes and is then called exactly like a metric function such as `f1_score`:

In [ ]:
class RetentionCost:
    """Total money lost to churn when we act on the model's flags."""

    def __init__(
        self, offer_cost: float = 50.0, customer_value: float = 600.0, acceptance_rate: float = 0.3
    ):
        self.offer_cost = offer_cost
        self.customer_value = customer_value
        self.acceptance_rate = acceptance_rate

    def __call__(self, y_true, y_pred) -> float:
        tn, fp, fn, tp = ConfusionMatrix.from_predictions(y_true, y_pred)
        offers = (tp + fp) * self.offer_cost
        lost_despite_offer = tp * (1 - self.acceptance_rate) * self.customer_value
        lost_unflagged = fn * self.customer_value
        return offers + lost_despite_offer + lost_unflagged

    def __repr__(self) -> str:
        return (
            f"RetentionCost(offer_cost={self.offer_cost}, "
            f"customer_value={self.customer_value}, acceptance_rate={self.acceptance_rate})"
        )


cost = RetentionCost()
print(cost)
print(f"Do nothing (flag nobody): {cost(y_test, np.zeros_like(y_test)):>10,.0f}")
for threshold in [0.5, 0.3, 0.2]:
    flagged = (churn_proba >= threshold).astype(int)
    print(f"Threshold {threshold}:            {cost(y_test, flagged):>10,.0f}")

Because `cost` is callable, any code written for metric functions accepts it unchanged. A dictionary of metrics can mix plain functions and our object:

In [ ]:
from sklearn.metrics import f1_score, recall_score

metrics = {"recall": recall_score, "f1": f1_score, "cost": cost}
for name, metric in metrics.items():
    print(f"{name:>6}: {metric(y_test, y_pred):,.2f}")

You meet this pattern all over the ecosystem: PyTorch layers are called as `layer(x)` (their `__call__` runs `forward`), and scikit-learn scorers made with `make_scorer` are callable objects.

## 6. Context managers: `__enter__` and `__exit__` (brief)

`with` guarantees that set-up and clean-up code runs, even if the block raises an error. You know it from `with open(path) as f:`. Writing one is two methods. A timer for model fitting is a small, honest use case:

In [ ]:
import time

from sklearn.ensemble import RandomForestClassifier


class Timer:
    """Measure how long the code inside a `with` block takes."""

    def __init__(self, label: str):
        self.label = label
        self.seconds = None

    def __enter__(self) -> "Timer":
        self._start = time.perf_counter()
        return self  # this is what `as t` receives

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        self.seconds = time.perf_counter() - self._start
        print(f"{self.label}: {self.seconds:.2f} s")
        return False  # do not swallow exceptions raised inside the block


with Timer("random forest, 200 trees") as t:
    RandomForestClassifier(n_estimators=200, random_state=42).fit(X_train, y_train)

## 7. When not to overload

Special methods make code shorter only when the meaning is obvious. Two rules of thumb:

- **Overload an operator only if every reader would guess the same meaning.** `cm1 + cm2` pooling counts: obvious. `model_a < model_b` comparing models: by accuracy? recall? cost? Readers cannot know. Use an explicit key instead: `best = max(results, key=lambda r: r.recall)`.
- **Keep the usual contracts.** `__len__` returns a non-negative int. `__eq__` is symmetric and returns `NotImplemented` for unknown types. `__repr__` never raises.

## Exercises

### Exercise 3.1 · Core: a CVResult that behaves like a sequence

`cross_val_score` returns a bare NumPy array of fold scores. Wrap it in a `CVResult` class so that:

- `CVResult(scores, metric="recall")` stores the scores (as a list or array) and the metric name,
- `len(result)` is the number of folds,
- `result[0]` is the score of the first fold,
- `for score in result:` iterates over the fold scores,
- `repr(result)` looks like `CVResult(recall: 0.534 ± 0.021 over 5 folds)` (mean and standard deviation, 3 decimals).

In [ ]:
from sklearn.model_selection import cross_val_score

fold_scores = cross_val_score(
    make_pipeline(StandardScaler(), LogisticRegression()), X_train, y_train, cv=5, scoring="recall"
)


class CVResult:
    def __init__(self, scores, metric: str):
        raise NotImplementedError("Your code here")


result = CVResult(fold_scores, metric="recall")
result

In [ ]:
# Check cell
assert len(result) == 5
assert result[0] == fold_scores[0]
assert list(result) == list(fold_scores)
assert repr(result) == (
    f"CVResult(recall: {np.mean(fold_scores):.3f} ± {np.std(fold_scores):.3f} over 5 folds)"
)
print("All checks passed")

<details><summary>Hint</summary>

Every method can delegate to the stored scores: `len(self.scores)`, `self.scores[i]`, `iter(self.scores)`. For the repr, use `np.mean` and `np.std` inside an f-string with `:.3f`.
</details>

<details><summary>Solution</summary>

```python
class CVResult:
    def __init__(self, scores, metric: str):
        self.scores = list(scores)
        self.metric = metric

    def __len__(self) -> int:
        return len(self.scores)

    def __getitem__(self, fold: int) -> float:
        return self.scores[fold]

    def __iter__(self):
        return iter(self.scores)

    def __repr__(self) -> str:
        return (f"CVResult({self.metric}: {np.mean(self.scores):.3f} "
                f"± {np.std(self.scores):.3f} over {len(self)} folds)")
```

Strictly speaking `__iter__` is optional here: if a class has `__getitem__` with integer indices, `for` falls back to calling it with 0, 1, 2, ... until `IndexError`. Writing `__iter__` explicitly is clearer.
</details>

### Exercise 3.2 · Stretch: pool the confusion matrices of five folds

Cross-validation gives you one confusion matrix per fold. Write `pooled_confusion(X, y, n_splits=5)` that:

1. loops over `StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42).split(X, y)`,
2. fits a fresh `make_pipeline(StandardScaler(), LogisticRegression())` on each training part,
3. computes a `ConfusionMatrix` on each validation part (threshold 0.5, i.e. `.predict`),
4. returns the pooled matrix (use `+`).

Then compare the **pooled recall** with the **mean of the per-fold recalls**. Are they the same? Why (not)?

In [ ]:
from sklearn.model_selection import StratifiedKFold


def pooled_confusion(X: pd.DataFrame, y: pd.Series, n_splits: int = 5) -> ConfusionMatrix:
    raise NotImplementedError("Your code here")


pooled = pooled_confusion(X_train, y_train)
pooled

In [ ]:
# Check cell
assert isinstance(pooled, ConfusionMatrix)
assert pooled.n == len(X_train)  # every customer is validated exactly once
assert pooled.tp + pooled.fn == y_train.sum()
print("All checks passed. Pooled recall:", round(pooled.recall, 3))

<details><summary>Hint</summary>

`split` yields pairs of integer position arrays: `for train_idx, val_idx in skf.split(X, y):` and then `X.iloc[train_idx]`. Start the pooling with the first fold's matrix, or keep a list and add them up in a loop.
</details>

<details><summary>Solution</summary>

```python
def pooled_confusion(X: pd.DataFrame, y: pd.Series, n_splits: int = 5) -> ConfusionMatrix:
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    pooled = None
    for train_idx, val_idx in skf.split(X, y):
        fold_model = make_pipeline(StandardScaler(), LogisticRegression())
        fold_model.fit(X.iloc[train_idx], y.iloc[train_idx])
        cm = ConfusionMatrix.from_predictions(y.iloc[val_idx], fold_model.predict(X.iloc[val_idx]))
        pooled = cm if pooled is None else pooled + cm
    return pooled
```

The pooled recall and the mean of fold recalls are close but usually not identical. The pooled value weights every churner equally; the mean weights every fold equally. With stratified folds of nearly equal size the difference is tiny. With very unequal batches (say a small December and a large November) it can matter, and pooling is usually the number you want to report.
</details>

### Exercise 3.3 · Challenge: make `sum()` work, and think about hashing

1. Run `sum([november, december])`. It fails, although `november + december` works. Explain why, using the table in section 1. (Hint: what is the start value of `sum`?)
2. Write a subclass `SummableConfusionMatrix(ConfusionMatrix)` that adds `__radd__` so that `sum(list_of_matrices)` returns the pooled matrix. Do not change `__add__`.
3. Discussion (write two or three sentences in a markdown cell): our `ConfusionMatrix` defines `__eq__` and is therefore unhashable. How would you make it hashable *correctly*? Why would it be a mistake to give a fitted model class an `__eq__` and `__hash__` based on its accuracy?

In [ ]:
class SummableConfusionMatrix(ConfusionMatrix):
    def __radd__(self, other):
        raise NotImplementedError("Your code here")


months = [
    SummableConfusionMatrix(tn=500, fp=120, fn=40, tp=110),
    SummableConfusionMatrix(tn=480, fp=130, fn=35, tp=125),
    SummableConfusionMatrix(tn=510, fp=100, fn=50, tp=105),
]
quarter = sum(months)
quarter

In [ ]:
# Check cell
assert quarter == ConfusionMatrix(tn=1490, fp=350, fn=125, tp=340)
assert sum(months[:1]) == months[0]
print("All checks passed")

<details><summary>Hint</summary>

`sum(items)` computes `0 + items[0] + items[1] + ...`. The first step is `0 + matrix`. `int.__add__` returns `NotImplemented` for a matrix, so Python tries `matrix.__radd__(0)`. Decide what adding 0 should mean.
</details>

<details><summary>Solution</summary>

```python
class SummableConfusionMatrix(ConfusionMatrix):
    def __radd__(self, other):
        if other == 0:  # the start value of sum()
            return self
        return self.__add__(other)
```

Alternative without `__radd__`: `sum(months, start=ConfusionMatrix(0, 0, 0, 0))`. That works but puts the burden on every caller.

On hashing: make the matrix immutable (a frozen dataclass, notebook 07) and hash the same fields that `__eq__` compares. Then equal matrices have equal hashes and nobody can change a matrix after it was put in a set. For a fitted model, equality by accuracy would declare two different models (different features, different coefficients) "equal", and a set of models would silently drop one of them. Models have identity, so keep the default identity-based `__eq__` and `__hash__`, and compare models explicitly with `max(models, key=...)`.
</details>

## Summary

- Special methods connect your class to Python syntax: `len()`, `[]`, `in`, `for`, `==`, `+`, calling with `()`, `with`.
- Always write `__repr__`. It is the cheapest debugging aid there is.
- Delegate: container methods are often one line that forwards to a list or DataFrame you already hold.
- Return `NotImplemented` (not `False`, not an exception) from `__eq__` and `__add__` for types you do not handle.
- `__call__` turns a configured object into a function: perfect for metrics with business parameters.
- Overload an operator only when its meaning is obvious; otherwise use a named method or a `key=` function.

**Common pitfalls**

- Defining `__eq__` and wondering why the object cannot go into a set (Python removed `__hash__`).
- `__str__` without `__repr__`: lists and error messages still show `<... at 0x...>`.
- Making `+` or `<` mean something surprising.

## Going further

Not part of the live session.

- **`__bool__`** to define truthiness explicitly, **`__contains__`** vs iteration fallback, **`__getitem__` with slices** (`batch[10:20]` receives a `slice` object).
- **Comparisons**: `__lt__`, `__le__`, ... and `functools.total_ordering`. Remember section 7 before using them.
- **`contextlib.contextmanager`** writes a context manager as a generator function with one `yield`, often shorter than a class.
- **`__format__`** controls f-string formatting such as `f"{cm:short}"`.
- The finished `ConfusionMatrix` and `RetentionCost` live in `src/churn/evaluation.py`.

Resources:

- Python data model reference: https://docs.python.org/3/reference/datamodel.html
- `contextlib` documentation: https://docs.python.org/3/library/contextlib.html
- Real Python, operator and function overloading: https://realpython.com/operator-function-overloading/